# dim_keyword_scd1 (SCD 1)

One row per normalized search keyword (`normalize_keyword`: lower-case, single spaces, diacritics kept). `keyword_key = xxhash64(keyword_normalized)`.

Task `dim_keyword_scd1` of the job "2. gold (poc tables)"; widgets `load_type` (incremental / full), `from_date`, `to_date` (Vietnam days, empty = yesterday .. tomorrow). The live dim is unchanged.

In [ ]:
%run ./_udfs

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))

# First load of the table = full (poc update_load_type: full once, then incremental)
if not spark.catalog.tableExists(f"{gold}.dim_keyword_scd1"):
    load_type = "full"
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
search_results = (spark.table(f"{silver}.stg_search_results")
                  .filter("search_keyword IS NOT NULL AND trim(search_keyword) <> ''")
                  .withColumn("keyword_normalized", F.expr("normalize_keyword(search_keyword)")))

## Incremental: keywords searched in the window

Their first spelling / variant count is then recomputed over **all** results.

In [ ]:
if load_type == "incremental":
    fetched_day = F.to_date(F.from_utc_timestamp("fetched_at", TZ))
    changed = (search_results
               .filter((fetched_day >= from_date) & (fetched_day < to_date))
               .select("keyword_normalized").distinct())
    search_results = search_results.join(changed, "keyword_normalized", "left_semi")

# Transformation

In [ ]:
from pyspark.sql.window import Window

per_keyword = Window.partitionBy("keyword_normalized")
first_searched = per_keyword.orderBy("fetched_at", "search_keyword")

result_df = (
    search_results
    .withColumn("rn", F.row_number().over(first_searched))
    .withColumn("raw_variant_count", F.size(F.collect_set("search_keyword").over(per_keyword)))
    .withColumn("first_searched_at", F.min("fetched_at").over(per_keyword))
    .filter("rn = 1")
    .select(
        F.xxhash64("keyword_normalized").alias("keyword_key"),
        F.col("keyword_normalized").cast("string").alias("keyword_normalized"),
        F.expr("unaccent_vi(keyword_normalized)").alias("keyword_unaccented"),
        F.col("search_keyword").cast("string").alias("keyword_raw_first"),
        F.col("raw_variant_count").cast("int").alias("raw_variant_count"),
        F.lit(None).cast("string").alias("dish"),   # not parsed yet
        F.lit(None).cast("string").alias("city"),
        F.col("first_searched_at").cast("timestamp").alias("first_searched_at"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["keyword_normalized"]
result_df = add_checksums(result_df, primary_key, exclude=["keyword_key"])

## Create table

In [ ]:
path = f"{gold}.dim_keyword_scd1"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        keyword_key BIGINT, keyword_normalized STRING, keyword_unaccented STRING, keyword_raw_first STRING,
        raw_variant_count INT, dish STRING, city STRING, first_searched_at TIMESTAMP, etl_date TIMESTAMP, etl_by STRING,
        primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Normalized search keyword dimension, SCD 1 (poc checksum method). keyword_key = xxhash64(keyword_normalized), -1 = Unknown.'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite (poc `full_refresh`). **incremental**: MERGE on `primary_checksum`; update only when `record_checksum` changed; insert new keys.

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Unknown row

Key −1, natural key NULL: facts with no match point to it. Inserted only when missing (no empty commit).

In [ ]:
if spark.table(path).where("keyword_key = -1").isEmpty():
    spark.sql(f"INSERT INTO {path} (keyword_key, etl_date, etl_by) VALUES (-1, current_timestamp(), 'Databricks')")

## Check

In [ ]:
dup = spark.sql(f"SELECT COUNT(*) - COUNT(DISTINCT keyword_normalized) FROM {path} WHERE keyword_normalized IS NOT NULL").first()[0]
assert dup == 0, f"{path}: {dup} duplicate keyword_normalized"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 2").where("operation <> 'WRITE'").select("operation", "operationMetrics").first()
print(f"✅ {path}: {spark.table(path).count()} rows", "·", last["operation"], dict(last["operationMetrics"] or {}) if last else "")